# DataFrame API Basics

{doc}`Introduction to DataFrames <../01_introduction_to_apache_spark/02_spark_dataframes_and_sql/01_introduction_to_dataframes>`
introduced the DataFrame, the difference between transformations and actions,
and the handful of methods you need to get started. This notebook builds on
that foundation with the everyday tools of real application code: inspecting
an unfamiliar table, deriving many columns at once, casting safely, writing
filters that do what you mean, conditional logic, strings and timestamps,
missing values, and removing duplicates.

All notebooks in this chapter run against Databricks serverless compute
through Databricks Connect, as set up in
{doc}`Setting Up Databricks Connect <../00_getting_started/01_setting_up_databricks_connect>`.

## The Bakehouse Dataset

Unity Catalog provides sample datasets in the `samples` catalog
([Sample datasets](https://docs.databricks.com/aws/en/discover/databricks-datasets#unity-catalog-datasets)).
This chapter uses `samples.bakehouse`, the sales records of a fictional chain
of bakery franchises:

| Table | Grain | Key columns |
| --- | --- | --- |
| `sales_transactions` | One row per sale | `transactionID`, `customerID`, `franchiseID` |
| `sales_customers` | One row per customer | `customerID` |
| `sales_franchises` | One row per bakery | `franchiseID`, `supplierID` |
| `sales_suppliers` | One row per ingredient supplier | `supplierID` |
| `media_customer_reviews` | One row per online review | `franchiseID` |

The tables are small enough that every query here returns in a few seconds,
but every technique on this page works unchanged on billions of rows.

:::{admonition} Unverified: Availability of `samples.bakehouse`
:class: warning

The `bakehouse` schema exists in the course's Free Edition workspace, as the
cells on these pages show, but the
[Sample datasets](https://docs.databricks.com/aws/en/discover/databricks-datasets#unity-catalog-datasets)
page doesn't list it among the schemas of the `samples` catalog. It couldn't
be confirmed in the official documentation that every workspace has it. If
your workspace doesn't, check the `samples` catalog in Catalog Explorer.
:::

Start a session and read the transactions table:

In [1]:
from pyspark.sql import functions as F

from project.utils.sparksession import get_spark_session

spark = get_spark_session(profile="free-edition")

transactions = spark.read.table("samples.bakehouse.sales_transactions")

## Inspecting a DataFrame

When you meet a new table, the first questions are what columns it has,
what types they are, and what the values look like. `printSchema` answers
the first two:

In [2]:
transactions.printSchema()

root
 |-- transactionID: long (nullable = true)
 |-- customerID: long (nullable = true)
 |-- franchiseID: long (nullable = true)
 |-- dateTime: timestamp (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unitPrice: long (nullable = true)
 |-- totalPrice: long (nullable = true)
 |-- paymentMethod: string (nullable = true)
 |-- cardNumber: long (nullable = true)



The same information is available programmatically through `columns`,
`dtypes` and `schema`, which is useful when code needs to adapt to the
columns it is given:

In [3]:
transactions.dtypes

[('transactionID', 'bigint'),
 ('customerID', 'bigint'),
 ('franchiseID', 'bigint'),
 ('dateTime', 'timestamp'),
 ('product', 'string'),
 ('quantity', 'bigint'),
 ('unitPrice', 'bigint'),
 ('totalPrice', 'bigint'),
 ('paymentMethod', 'string'),
 ('cardNumber', 'bigint')]

`show` prints rows. Two options make it far more readable on wide tables:
`truncate=False` stops long values from being cut off at 20 characters, and
`vertical=True` prints one block per row instead of one very wide line.

In [4]:
transactions.show(2, vertical=True)

-RECORD 0-----------------------------
 transactionID | 1002961              
 customerID    | 2000253              
 franchiseID   | 3000047              
 dateTime      | 2024-05-14 12:17:... 
 product       | Golden Gate Ginger   
 quantity      | 8                    
 unitPrice     | 3                    
 totalPrice    | 24                   
 paymentMethod | amex                 
 cardNumber    | 378154478982993      
-RECORD 1-----------------------------
 transactionID | 1003007              
 customerID    | 2000226              
 franchiseID   | 3000047              
 dateTime      | 2024-05-10 23:10:... 
 product       | Austin Almond Bis... 
 quantity      | 36                   
 unitPrice     | 3                    
 totalPrice    | 108                  
 paymentMethod | mastercard           
 cardNumber    | 2244626981238094     
only showing top 2 rows


To get a feel for the *values*, `describe` computes basic statistics for
numeric and string columns. `summary` by default computes the count, mean,
standard deviation, minimum, approximate quartiles and maximum
([Apache Spark API reference: DataFrame.summary](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.summary.html)):

In [5]:
transactions.select("quantity", "unitPrice", "totalPrice").summary().show()

+-------+-----------------+---------+------------------+
|summary|         quantity|unitPrice|        totalPrice|
+-------+-----------------+---------+------------------+
|  count|             3333|     3333|              3333|
|   mean|6.647764776477648|      3.0|19.943294329432945|
| stddev|6.544484308501296|      0.0|  19.6334529255051|
|    min|                1|        3|                 3|
|    25%|                3|        3|                 9|
|    50%|                6|        3|                18|
|    75%|                9|        3|                27|
|    max|               60|        3|               180|
+-------+-----------------+---------+------------------+



Every transaction sells a product at a unit price of 3, quantities range from
1 to 60, and the quartiles show that half of all sales are between
3 and 9 items.

:::{admonition} Note: `show`, `describe` and `summary` Are Actions
:class: note

Each of these runs a Spark job. That is fine while exploring, but in a
pipeline every stray `show` re-runs the plan, as explained in
{doc}`Introduction to DataFrames <../01_introduction_to_apache_spark/02_spark_dataframes_and_sql/01_introduction_to_dataframes>`.
Remove them, or guard them behind a debug flag, before the code goes to
production.
:::

## Deriving Columns

`withColumn` adds or replaces a single column. When you derive several
columns, `withColumns` takes a dictionary of column names and expressions and
adds or replaces them all in one step
([Apache Spark API reference: DataFrame.withColumns](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumns.html)).
Each `withColumn` call adds a projection to the plan, so calling it in a loop
can generate big plans that cause performance problems
([Apache Spark API reference: DataFrame.withColumn](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumn.html)):

In [6]:
enriched = transactions.withColumns(
    {
        "sale_date": F.to_date("dateTime"),
        "sale_hour": F.hour("dateTime"),
        "weekday": F.date_format("dateTime", "EEEE"),
        "is_bulk_order": F.col("quantity") >= 30,
    }
)

enriched.select("transactionID", "dateTime", "sale_date", "sale_hour", "weekday", "is_bulk_order").show(5, truncate=False)

+-------------+--------------------------+----------+---------+--------+-------------+
|transactionID|dateTime                  |sale_date |sale_hour|weekday |is_bulk_order|
+-------------+--------------------------+----------+---------+--------+-------------+
|1002961      |2024-05-14 12:17:01.495952|2024-05-14|12       |Tuesday |false        |
|1003007      |2024-05-10 23:10:10.239954|2024-05-10|23       |Friday  |true         |
|1003017      |2024-05-16 16:34:10.61372 |2024-05-16|16       |Thursday|true         |
|1003068      |2024-05-02 04:31:51.612094|2024-05-02|4        |Thursday|false        |
|1003103      |2024-05-04 23:44:26.902224|2024-05-04|23       |Saturday|false        |
+-------------+--------------------------+----------+---------+--------+-------------+
only showing top 5 rows


Its siblings `withColumnsRenamed` and `drop` rename and remove several
columns at once. Here we tidy the names into `snake_case` and drop the card
number, which nothing downstream should ever need:

In [7]:
tidy = enriched.withColumnsRenamed(
    {
        "transactionID": "transaction_id",
        "customerID": "customer_id",
        "franchiseID": "franchise_id",
        "unitPrice": "unit_price",
        "totalPrice": "total_price",
        "paymentMethod": "payment_method",
    }
).drop("cardNumber", "dateTime")

tidy.columns

['transaction_id',
 'customer_id',
 'franchise_id',
 'product',
 'quantity',
 'unit_price',
 'total_price',
 'payment_method',
 'sale_date',
 'sale_hour',
 'weekday',
 'is_bulk_order']

`F.lit` turns a Python value into a column, for when every row should get
the same value:

In [8]:
tidy.select("transaction_id", F.lit("USD").alias("currency")).show(3)

+--------------+--------+
|transaction_id|currency|
+--------------+--------+
|       1002961|     USD|
|       1003007|     USD|
|       1003017|     USD|
+--------------+--------+
only showing top 3 rows


## Data Types and Casting

`cast` converts a column to another type. You can name the target type with
a DDL string such as `"int"`, `"decimal(10,2)"` or `"date"`, or with a type
object from `pyspark.sql.types`:

In [9]:
tidy.select(
    "total_price",
    F.col("total_price").cast("decimal(10,2)").alias("as_decimal"),
    F.col("total_price").cast("string").alias("as_string"),
).show(3)

+-----------+----------+---------+
|total_price|as_decimal|as_string|
+-----------+----------+---------+
|         24|     24.00|       24|
|        108|    108.00|      108|
|        120|    120.00|      120|
+-----------+----------+---------+
only showing top 3 rows


Spark 4 runs in **ANSI mode** by default
([Apache Spark: Spark Release 4.0.0](https://spark.apache.org/releases/spark-release-4-0-0.html)),
and so does Databricks serverless compute
([Serverless compute limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations)).
In ANSI mode an invalid cast throws a runtime exception instead of returning
a silent `NULL`
([ANSI compliance in Databricks Runtime](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-ansi-compliance#cast)):

In [10]:
spark.conf.get("spark.sql.ansi.enabled")

'true'

In [11]:
raw_quantities = spark.createDataFrame(
    [("12",), ("7",), ("a dozen",)],
    schema="quantity STRING",
)

raw_quantities.select(F.col("quantity").cast("int")).show()

{"ts": "2026-10-12 22:44:55.570", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value 'a dozen' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"cast\" was called from\n<command-None>, line 6 in cell [12]\n\n\nJVM stacktrace:\norg.apache.spark.SparkNumberFormatException\n\tat org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNumberError(QueryExecutionErrors.scala:205)\n\tat org.apache.spark.sql.catalyst.util.UTF8StringUtils$.withException(UTF8StringUtils.scala:57)\n\tat org.apache.spark.sql.catalyst.util.UTF8StringUtils$.toIntExact(UTF8StringUtils.scala:34)\n\tat org.apache.spark.sql.catalyst.expressions.Cast.$anonfun$castToInt$2(Cast.scala:1618)\n\tat org.apache.spark.sql.catalyst.expressions.Cast.$anonfun$castToInt$2$adapted(Cast

NumberFormatException: [CAST_INVALID_INPUT] The value 'a dozen' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
<command-None>, line 6 in cell [12]


JVM stacktrace:
org.apache.spark.SparkNumberFormatException
	at org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNumberError(QueryExecutionErrors.scala:205)
	at org.apache.spark.sql.catalyst.util.UTF8StringUtils$.withException(UTF8StringUtils.scala:57)
	at org.apache.spark.sql.catalyst.util.UTF8StringUtils$.toIntExact(UTF8StringUtils.scala:34)
	at org.apache.spark.sql.catalyst.expressions.Cast.$anonfun$castToInt$2(Cast.scala:1618)
	at org.apache.spark.sql.catalyst.expressions.Cast.$anonfun$castToInt$2$adapted(Cast.scala:1618)
	at org.apache.spark.sql.catalyst.expressions.Cast.buildCast(Cast.scala:1216)
	at org.ap

The error class, `CAST_INVALID_INPUT`, names the value that couldn't be
converted, `'a dozen'`, and suggests `try_cast`.

When bad values are expected and should become `NULL` instead, use
`try_cast`, which returns `NULL` for a value that can't be converted
([try_cast function](https://docs.databricks.com/aws/en/sql/language-manual/functions/try_cast)).
The `try_` prefix marks the forgiving variant of other functions too, such as
`try_divide` and `try_add`
([ANSI compliance in Databricks Runtime](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-ansi-compliance#useful-functions-for-ansi-mode)):

In [12]:
raw_quantities.select("quantity", F.col("quantity").try_cast("int").alias("parsed")).show()

+--------+------+
|quantity|parsed|
+--------+------+
|      12|    12|
|       7|     7|
| a dozen|  NULL|
+--------+------+



## Filtering Rows

`filter` (and its alias `where`) keeps the rows for which a boolean column
expression is true. Besides the comparison operators, `Column` has methods
for the most common tests; `between` includes both bounds
([Apache Spark API reference: Column.between](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Column.between.html)):

| Expression | Keeps rows where |
| --- | --- |
| `F.col("x").isin("a", "b")` | `x` is one of the listed values |
| `F.col("x").between(1, 10)` | `1 <= x <= 10` (both ends inclusive) |
| `F.col("x").like("Golden%")` | `x` matches a SQL `LIKE` pattern |
| `F.col("x").rlike("^[A-Z]")` | `x` matches a regular expression |
| `F.col("x").startswith("Go")` | `x` starts with a prefix |
| `F.col("x").isNull()` / `isNotNull()` | `x` is, or is not, missing |

Conditions are combined with `&` (and), `|` (or) and `~` (not), **not** with
Python's `and`, `or` and `not`. Because `&` and `|` bind more tightly than
comparisons in Python, every comparison must be wrapped in parentheses:

In [13]:
large_card_sales = tidy.filter(
    (F.col("quantity") >= 30)
    & F.col("payment_method").isin("visa", "mastercard")
    & ~F.col("product").startswith("Golden")
)

large_card_sales.count()

33

(filter-parentheses)=
Forgetting the parentheses does not give you a wrong answer. It fails
outright, because Python evaluates `30 & F.col(...)` first and then tries to
turn a `Column` into a Python `bool`:

In [14]:
tidy.filter(F.col("quantity") >= 30 & F.col("payment_method") == "visa")

PySparkValueError: [CANNOT_CONVERT_COLUMN_INTO_BOOL] Cannot convert column into bool: please use '&' for 'and', '|' for 'or', '~' for 'not' when building DataFrame boolean expressions.

The error is raised by the Python client before anything reaches the
server, and its message points at the fix: use `&`, `|` and `~`, with
parentheses around each comparison.

A filter can also be a SQL string, which is often shorter for simple
conditions:

In [15]:
tidy.filter("quantity >= 30 AND payment_method IN ('visa', 'mastercard')").count()

40

## Conditional Logic with `when`

`F.when(condition, value)` is the DataFrame version of SQL's `CASE WHEN`.
Chain further `when` calls for more branches and finish with `otherwise`;
the first condition that matches wins. Without `otherwise`, unmatched rows
get `NULL`
([Apache Spark API reference: functions.when](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.when.html)).

In [16]:
sized = tidy.withColumn(
    "order_size",
    F.when(F.col("quantity") >= 30, "large")
    .when(F.col("quantity") >= 10, "medium")
    .otherwise("small"),
)

sized.select("transaction_id", "quantity", "order_size").show(5)

+--------------+--------+----------+
|transaction_id|quantity|order_size|
+--------------+--------+----------+
|       1002961|       8|     small|
|       1003007|      36|     large|
|       1003017|      40|     large|
|       1003068|      28|    medium|
|       1003103|      28|    medium|
+--------------+--------+----------+
only showing top 5 rows


:::::{dropdown} Example: How Many Orders Are Large?
:open:
:color: primary
:icon: pencil
:class-body: example-body

Using the `order_size` column from above, what share of the 3,333
transactions are large orders of 30 items or more?

**Count the large orders.**

```python
large = sized.filter(F.col("order_size") == "large").count()
total = sized.count()
```

**Divide.**

$$
\frac{\text{large orders}}{\text{all orders}} = \frac{66}{3333} \approx 0.020
$$

Only about 2% of sales are large orders. Most customers buy a handful of
items: the median quantity, from the `summary` output above, is 6.
:::::

Checking the hand calculation against Spark:

In [17]:
large = sized.filter(F.col("order_size") == "large").count()
total = sized.count()

large, total, round(large / total, 3)

(66, 3333, 0.02)

## Strings and Timestamps

`pyspark.sql.functions` contains a large library of string and date
functions. A few of the most useful ones, applied to the bakehouse data:

In [18]:
tidy.select(
    "product",
    F.upper("product").alias("upper"),
    F.regexp_replace("product", r"\s+", "_").alias("slug"),
    F.split("product", " ").getItem(0).alias("first_word"),
    F.concat_ws(" / ", "product", "payment_method").alias("label"),
).show(5, truncate=False)

+----------------------+----------------------+----------------------+----------+-----------------------------------+
|product               |upper                 |slug                  |first_word|label                              |
+----------------------+----------------------+----------------------+----------+-----------------------------------+
|Golden Gate Ginger    |GOLDEN GATE GINGER    |Golden_Gate_Ginger    |Golden    |Golden Gate Ginger / amex          |
|Austin Almond Biscotti|AUSTIN ALMOND BISCOTTI|Austin_Almond_Biscotti|Austin    |Austin Almond Biscotti / mastercard|
|Austin Almond Biscotti|AUSTIN ALMOND BISCOTTI|Austin_Almond_Biscotti|Austin    |Austin Almond Biscotti / mastercard|
|Pearly Pies           |PEARLY PIES           |Pearly_Pies           |Pearly    |Pearly Pies / amex                 |
|Pearly Pies           |PEARLY PIES           |Pearly_Pies           |Pearly    |Pearly Pies / visa                 |
+----------------------+----------------------+---------

For timestamps, `date_trunc` rounds down to a unit such as an hour or a
week, `date_add` and `datediff` do calendar arithmetic, and `date_format`
renders a timestamp with a
[datetime pattern](https://spark.apache.org/docs/latest/sql-ref-datetime-pattern.html):

In [19]:
enriched.select(
    "dateTime",
    F.date_trunc("hour", "dateTime").alias("hour_bucket"),
    F.date_add("sale_date", 7).alias("one_week_later"),
    F.datediff(F.lit("2024-06-01").cast("date"), "sale_date").alias("days_until_june"),
    F.date_format("dateTime", "yyyy-MM-dd HH:mm").alias("formatted"),
).show(5, truncate=False)

+--------------------------+-------------------+--------------+---------------+----------------+
|dateTime                  |hour_bucket        |one_week_later|days_until_june|formatted       |
+--------------------------+-------------------+--------------+---------------+----------------+
|2024-05-14 12:17:01.495952|2024-05-14 12:00:00|2024-05-21    |18             |2024-05-14 12:17|
|2024-05-10 23:10:10.239954|2024-05-10 23:00:00|2024-05-17    |22             |2024-05-10 23:10|
|2024-05-16 16:34:10.61372 |2024-05-16 16:00:00|2024-05-23    |16             |2024-05-16 16:34|
|2024-05-02 04:31:51.612094|2024-05-02 04:00:00|2024-05-09    |30             |2024-05-02 04:31|
|2024-05-04 23:44:26.902224|2024-05-04 23:00:00|2024-05-11    |28             |2024-05-04 23:44|
+--------------------------+-------------------+--------------+---------------+----------------+
only showing top 5 rows


:::{admonition} Note: Session Time Zone
:class: note

A `TIMESTAMP` is an absolute point in time. It is normalised and stored in
UTC, and the session time zone is applied when it is read
([TIMESTAMP type](https://docs.databricks.com/aws/en/sql/language-manual/data-types/timestamp-type)).
On serverless compute the session time zone, `spark.sql.session.timeZone`,
defaults to `Etc/UTC` and is one of the few properties you can set
([Set Spark configuration properties on Databricks](https://docs.databricks.com/aws/en/spark/conf#configure-spark-properties-for-serverless-notebooks-and-jobs)).
Functions like `hour` and `to_date` also work in the session time zone, so
the same row can land on a different date for two users with different
session settings. Set the time zone explicitly in pipelines whose results
depend on calendar boundaries.
:::

## Missing Values

The bakehouse tables happen to have no missing values, so we build a small
DataFrame that does:

In [20]:
reviews = spark.createDataFrame(
    [
        ("Pearly Pies", 5, "Loved it"),
        ("Orchard Oasis", None, "No rating given"),
        ("Tokyo Tidbits", 3, None),
        (None, 4, "Forgot the product"),
    ],
    schema="product STRING, stars INT, comment STRING",
)

reviews.show()

+-------------+-----+------------------+
|      product|stars|           comment|
+-------------+-----+------------------+
|  Pearly Pies|    5|          Loved it|
|Orchard Oasis| NULL|   No rating given|
|Tokyo Tidbits|    3|              NULL|
|         NULL|    4|Forgot the product|
+-------------+-----+------------------+



A `NULL` means "unknown", and Spark follows SQL's three-valued logic: a
comparison with a `NULL` operand is itself `NULL`, and a filter keeps a row
only when its condition is `True`
([Apache Spark: NULL Semantics](https://spark.apache.org/docs/latest/sql-ref-null-semantics.html)).
This is a frequent source of silently lost rows. The filter below looks like
"everything except Pearly Pies" but also drops the row with no product:

In [21]:
reviews.filter(F.col("product") != "Pearly Pies").show()

+-------------+-----+---------------+
|      product|stars|        comment|
+-------------+-----+---------------+
|Orchard Oasis| NULL|No rating given|
|Tokyo Tidbits|    3|           NULL|
+-------------+-----+---------------+



To keep the unknown rows, say so explicitly:

In [22]:
reviews.filter((F.col("product") != "Pearly Pies") | F.col("product").isNull()).show()

+-------------+-----+------------------+
|      product|stars|           comment|
+-------------+-----+------------------+
|Orchard Oasis| NULL|   No rating given|
|Tokyo Tidbits|    3|              NULL|
|         NULL|    4|Forgot the product|
+-------------+-----+------------------+



The `na` accessor replaces or removes missing values. `fillna` takes a
dictionary of per-column defaults, and `dropna` removes rows with missing
values, optionally only in the columns listed in `subset`:

In [23]:
reviews.fillna({"stars": 0, "comment": "(none)"}).show()

+-------------+-----+------------------+
|      product|stars|           comment|
+-------------+-----+------------------+
|  Pearly Pies|    5|          Loved it|
|Orchard Oasis|    0|   No rating given|
|Tokyo Tidbits|    3|            (none)|
|         NULL|    4|Forgot the product|
+-------------+-----+------------------+



In [24]:
reviews.dropna(subset=["product", "stars"]).show()

+-------------+-----+--------+
|      product|stars| comment|
+-------------+-----+--------+
|  Pearly Pies|    5|Loved it|
|Tokyo Tidbits|    3|    NULL|
+-------------+-----+--------+



`F.coalesce` returns the first non-null value among its arguments, which is
handy for falling back from one column to another:

In [25]:
reviews.select("product", F.coalesce("comment", "product", F.lit("?")).alias("display_text")).show()

+-------------+------------------+
|      product|      display_text|
+-------------+------------------+
|  Pearly Pies|          Loved it|
|Orchard Oasis|   No rating given|
|Tokyo Tidbits|     Tokyo Tidbits|
|         NULL|Forgot the product|
+-------------+------------------+



## Sorting and Limiting

`orderBy` (alias `sort`) accepts column names or `Column` expressions.
`F.desc` reverses the order, and the `asc_nulls_last` family controls where
`NULL`s go:

In [26]:
reviews.orderBy(F.col("stars").desc_nulls_last(), "product").show()

+-------------+-----+------------------+
|      product|stars|           comment|
+-------------+-----+------------------+
|  Pearly Pies|    5|          Loved it|
|         NULL|    4|Forgot the product|
|Tokyo Tidbits|    3|              NULL|
|Orchard Oasis| NULL|   No rating given|
+-------------+-----+------------------+



Sorting a whole DataFrame is a wide transformation that shuffles every row
(see
{doc}`Distributed Systems Programming Fundamentals <../01_introduction_to_apache_spark/03_distributed_systems_programming_fundamentals/03_distributed_systems_programming_fundamentals>`).
`orderBy(...).limit(n)` is much cheaper, though, because Spark recognises
the pattern and plans a *top-N* operator instead of a full sort:

In [27]:
top3 = tidy.orderBy(F.desc("total_price")).limit(3).select("transaction_id", "product", "quantity", "total_price")
top3.show()
top3.explain()

+--------------+--------------------+--------+-----------+
|transaction_id|             product|quantity|total_price|
+--------------+--------------------+--------+-----------+
|       1001320|Austin Almond Bis...|      60|        180|
|       1001783|         Pearly Pies|      60|        180|
|       1002065|     Outback Oatmeal|      60|        180|
+--------------+--------------------+--------+-----------+



== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonTopK(sortOrder=[total_price#20374L DESC NULLS LAST], partitionOrderCount=0)
      +- PhotonProject [transactionID#20350L AS transaction_id#20370L, product#20354, quantity#20355L, totalPrice#20357L AS total_price#20374L]
         +- PhotonScan parquet samples.bakehouse.sales_transactions[transactionID#20350L,product#20354,quantity#20355L,totalPrice#20357L] DataFilters: [], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://system-tables-prod-us-east-2-uc-metastore-bucket/metastore/b..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<transactionID:bigint,product:string,quantity:bigint,totalPrice:bigint>, RequiredDataFilters: []


== Photon Explanation ==
The query is fully supported by Photon.
== Optimizer Statistics (table names per statistics state) ==
  missing = sales_transactions
  partial = 
  full    = 
Corrective actions: consider running the following c

The plan has no full sort: a single `PhotonTopK` operator keeps only the 3
largest rows as the data streams past.

## Removing Duplicates

`distinct` removes rows that are identical in every column. To find the
unique combinations of a few columns, select them first:

In [28]:
tidy.select("product", "payment_method").distinct().count()

18

`dropDuplicates` takes a list of columns and keeps one row per unique
combination of *those* columns, with all the other columns intact. Its
documentation doesn't say which row of each group survives
([Apache Spark API reference: DataFrame.dropDuplicates](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.dropDuplicates.html)).
When it matters, use a window function as shown in
{doc}`ETL with the DataFrame API <../01_introduction_to_apache_spark/04_etl_with_the_dataframe_api/index>`.

In [29]:
first_seen_customers = tidy.dropDuplicates(["customer_id"])

first_seen_customers.count()

300

## Putting It Together

A typical cleaning step combines everything above into one function that
takes and returns a DataFrame, so it can be tested and reused:

In [30]:
from pyspark.sql import DataFrame


def clean_transactions(df: DataFrame) -> DataFrame:
    return (
        df.withColumnsRenamed(
            {
                "transactionID": "transaction_id",
                "customerID": "customer_id",
                "franchiseID": "franchise_id",
                "unitPrice": "unit_price",
                "totalPrice": "total_price",
                "paymentMethod": "payment_method",
            }
        )
        .withColumns(
            {
                "sale_date": F.to_date("dateTime"),
                "sale_hour": F.hour("dateTime"),
                "order_size": F.when(F.col("quantity") >= 30, "large")
                .when(F.col("quantity") >= 10, "medium")
                .otherwise("small"),
            }
        )
        .filter((F.col("quantity") > 0) & (F.col("total_price") > 0))
        .dropDuplicates(["transaction_id"])
        .drop("cardNumber")
    )


clean = clean_transactions(transactions)
clean.show(5)

+--------------+-----------+------------+--------------------+--------------------+--------+----------+-----------+--------------+----------+---------+----------+
|transaction_id|customer_id|franchise_id|            dateTime|             product|quantity|unit_price|total_price|payment_method| sale_date|sale_hour|order_size|
+--------------+-----------+------------+--------------------+--------------------+--------+----------+-----------+--------------+----------+---------+----------+
|       1002961|    2000253|     3000047|2024-05-14 12:17:...|  Golden Gate Ginger|       8|         3|         24|          amex|2024-05-14|       12|     small|
|       1003007|    2000226|     3000047|2024-05-10 23:10:...|Austin Almond Bis...|      36|         3|        108|    mastercard|2024-05-10|       23|     large|
|       1003017|    2000108|     3000047|2024-05-16 16:34:...|Austin Almond Bis...|      40|         3|        120|    mastercard|2024-05-16|       16|     large|
|       1003068|    20

## Choosing Between Similar Tools

Several pairs of methods on this page overlap. The choice usually comes down
to one question:

| Need | Use | Instead of |
| --- | --- | --- |
| Add or replace several columns | `withColumns({...})` or one `select` | `withColumn` in a loop |
| Convert values that may be malformed | `try_cast` (bad values become `NULL`) | `cast`, which fails the query under ANSI mode |
| A short, fixed condition | A SQL string in `filter("...")` | Column expressions, which are better when the condition is built in code |
| One row per key, any row will do | `dropDuplicates([...])` | A window function, which you need when the surviving row matters |
| Replace missing values | `fillna` with a per-column dictionary | `F.coalesce`, which you need when the fallback is another column |

## Common Mistakes

- **Combining conditions with `and`, `or`, `not`, or without parentheses.**
  Use `&`, `|` and `~`, and wrap every comparison in parentheses, as the
  error {ref}`above <filter-parentheses>` showed.
- **Comparing with `NULL` using `==` or `!=`.** The comparison returns `NULL`
  and the filter drops the row. Test with `isNull()` / `isNotNull()`, or use
  the null-safe `eqNullSafe` (SQL `<=>`)
  ([Apache Spark: NULL Semantics](https://spark.apache.org/docs/latest/sql-ref-null-semantics.html)).
- **Casting dirty strings with `cast`.** Under ANSI mode one malformed value
  fails the whole query. Use `try_cast` and then count the `NULL`s it
  produced
  ([try_cast function](https://docs.databricks.com/aws/en/sql/language-manual/functions/try_cast)).
- **Leaving `show` and `count` calls in pipeline code.** Each one is an action
  that runs the plan again, as explained in
  {doc}`Introduction to DataFrames <../01_introduction_to_apache_spark/02_spark_dataframes_and_sql/01_introduction_to_dataframes>`.
- **Assuming the session time zone.** Date boundaries depend on
  `spark.sql.session.timeZone`
  ([TIMESTAMP type](https://docs.databricks.com/aws/en/sql/language-manual/data-types/timestamp-type)).
  Set it explicitly when results depend on them.

Next, {doc}`Grouping and Aggregating Data <02_grouping_and_aggregating_data>`
turns these cleaned rows into summaries.

## Resources

- [PySpark functions reference (Apache Spark)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)
- [ANSI compliance in Spark SQL (Apache Spark)](https://spark.apache.org/docs/latest/sql-ref-ansi-compliance.html)
- [Datetime patterns (Apache Spark)](https://spark.apache.org/docs/latest/sql-ref-datetime-pattern.html)
- [NULL semantics (Apache Spark)](https://spark.apache.org/docs/latest/sql-ref-null-semantics.html)